# Phase 1: Environment Setup & Modular Ingestion
Importing custom data pipelines from the src/ directory to maintain clean orchestration. Loading the serialized processed audio array data saved on the hard disk into the notebook. Confimation of shape to ensure data is properly loaded

In [ ]:
import os
import sys
import numpy as np

# Point to the src folder so Jupyter can find our custom libraries
sys.path.append("..")
from src.data_utils import prepare_data
from src.data_augment import augment_training_data
%load_ext autoreload
%autoreload 2

# Load the serialized data from the directory where they were saved
DATA_DIR = os.path.join("..", "data", "processed")
X = np.load(os.path.join(DATA_DIR, "X_features.npy"))
y = np.load(os.path.join(DATA_DIR, "y_labels.npy"))
actors = np.load(os.path.join(DATA_DIR, "actor_ids.npy"))

print(f"Loaded X shape: {X.shape}")
print(f"Loaded y shape: {y.shape}")
print(f"Loaded actors shape: {actors.shape}")

# Phase 2: Subject-Independent Data Split and Data Preparation
This handles Subject-Independent Cross-Validation (splitting by actor ID), 
string-to-categorical label encoding, feature standardization (scaling) to prevent data leakage, 
and tensor reshaping to add the necessary channel dimension for 2D convolutions. 
It also serializes the fitted encoders and scalers to disk for future inference on raw data.

In [ ]:

X_train_final = np.array([])  # Clear the variable to prevent inflated data due to cell re-execution
X_test_final = np.array([])  # Clear the variable to prevent inflated data due to cell re-execution
y_train = np.array([])  # Clear the original training labels to prevent inflated data due to cell re-execution
y_test = np.array([])  # Clear the original test labels to free up memory and prevent inflated data due to cell re-execution
X_train_final, X_test_final, y_train, y_test, num_classes = prepare_data(X, y, actors)

print("\n--- DEEP LEARNING TENSORS READY ---")
print(f"Final X_train shape for CNN: {X_train_final.shape}")
print(f"Final X_test shape for CNN: {X_test_final.shape}")
print(f"Final y_train shape for CNN: {y_train.shape}")
print(f"Final y_test shape for CNN: {y_test.shape}")
print(f"Number of unique actors in the dataset: {len(np.unique(actors))}")
print(f"Total number of target classes: {num_classes}")

# Phase 3: Data Augmenting
Doubles the training data by appending a SpecAugmented copy of every sample with a max frequency and time mask

In [ ]:
# Double the training data using SpecAugment
X_train_final, y_train = augment_training_data(X_train_final, y_train, freq_mask_param=7, time_mask_param=15)

print(f"Augmented X_train shape for CNN: {X_train_final.shape}")
print(f"Augmented y_train shape for CNN: {y_train.shape}")

In [ ]:
# Define the absolute path to your models folder
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True) # Creates the folder if it somehow gets deleted


# Phase 4 (for v9): Spatially-Aligned CRNN & Dynamic Focal Loss

In this stage, we construct the core Convolutional Recurrent Neural Network (CRNN) optimized for edge deployment. Key architectural implementations include:

* **Targeted Alpha Weights & Dynamic Focal Loss:** A custom, serializable loss function (`DynamicFocalLoss`) is deployed to handle severe acoustic overlaps (e.g., Calm vs. Neutral) in the RAVDESS dataset. A dynamic Gamma callback slowly increases the focusing parameter to prevent gradient explosion early in training.
* **Spatial Tensor Alignment (The `Permute` Fix):** A `Permute((2, 1, 3))` layer is injected immediately before the `Reshape` layer. This prevents C-contiguous memory scrambling by explicitly aligning the spatial dimensions to `(Time, Frequency, Channels)`, allowing the LSTM to accurately read the chronological sequence of the audio.
* **Parameter Optimization:** Heavy $L_2$ regularization was stripped from the Convolutional blocks to prevent parameter starvation, relying instead on Spatial Dropout and Batch Normalization to bridge the generalization gap.

In [ ]:
import os
import tensorflow as tf
import joblib
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, BatchNormalization, Dense, Reshape, LSTM, Permute
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalFocalCrossentropy, Loss
import numpy as np



# Assume X_train_final, y_train, X_test_final, y_test are loaded from your data_loader
# input_shape = (60, 125, 1)

# --- 1. TARGETED ALPHA ARRAY (FOCAL LOSS ONLY) ---
# We removed class_weights from model.fit() to prevent the "Double-Steering" gradient explosion.
# These alphas will gently balance the classes.
classes = ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
alpha_weights = [0.125] * 8 
alpha_weights[classes.index('fearful')] = 0.20  # Gentle nudge for Fearful
alpha_weights[classes.index('disgust')] = 0.05  # Lower penalty
print(f"Targeted Alpha Array: {alpha_weights}")

# --- 2. CUSTOM SERIALIZABLE FOCAL LOSS ---
@tf.keras.utils.register_keras_serializable()
class DynamicFocalLoss(CategoricalFocalCrossentropy):
    def __init__(self, targeted_alpha=None, gamma=0.5, **kwargs):
        super().__init__(alpha=targeted_alpha, gamma=gamma, **kwargs)
        self.dynamic_gamma = tf.Variable(gamma, dtype=tf.float32, trainable=False)
        self.targeted_alpha = targeted_alpha

    def call(self, y_true, y_pred):
        self.gamma = self.dynamic_gamma
        return super().call(y_true, y_pred)

    def get_config(self):
        config = super().get_config()
        config.update({
            "targeted_alpha": self.targeted_alpha,
            "gamma": float(self.dynamic_gamma.numpy())
        })
        return config

# --- 3. DYNAMIC GAMMA CALLBACK ---
class DynamicGammaCallback(Callback):
    def on_epoch_begin(self, epoch, logs=None):
        new_gamma = min(0.5 + (epoch * 0.1), 3.0) # Peaks at 3.0 to heavily focus on hard overlaps
        if hasattr(self.model.loss, 'dynamic_gamma'):
            tf.keras.backend.set_value(self.model.loss.dynamic_gamma, new_gamma)

# --- 4. OPTIMIZED CRNN ARCHITECTURE ---
print("Building the Optimized CRNN model...")
model = Sequential()

# Block 1 (Removed L2 to prevent parameter starvation)
model.add(Conv2D(32, kernel_size=(3, 3), padding='same', activation='relu', input_shape=(60, 125, 1)))
model.add(BatchNormalization()) 
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3)) 

# Block 2
model.add(Conv2D(64, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) 
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3)) # Slightly lowered from 0.4 to retain more features

# Block 3
model.add(Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) 
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

# Block 4
model.add(Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) 
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

# --- THE SPATIAL ALIGNMENT FIX ---
# Output shape is currently (None, 3, 7, 128) -> (Batch, Freq, Time, Channels)
# We MUST permute to swap Freq and Time BEFORE flattening to preserve the sequence!
model.add(Permute((2, 1, 3))) # Shape becomes (None, 7, 3, 128)
model.add(Reshape((7, 384)))  # Correctly unrolls 7 time steps with 384 features each

# LSTM Sequence Processor (Kept recurrent L2 to stabilize temporal learning)
model.add(LSTM(64, return_sequences=False, recurrent_regularizer=l2(0.01), recurrent_dropout=0.3))
model.add(Dropout(0.4))

# Output
model.add(Dense(8, activation='softmax'))

# Compile
model.compile(
    loss=DynamicFocalLoss(targeted_alpha=alpha_weights), 
    optimizer=Adam(learning_rate=0.001), # Reset LR to standard 1e-3 to explore the new landscape
    metrics=['accuracy']
)

model.summary()

# Phase 5 Execution (for v9): Callback Orchestration & Model Training

In this execution stage, we implement a robust suite of MLOps callbacks to tightly control the training loop, ensuring optimal convergence and preventing catastrophic overfitting:

* **Optimizer Health (`ReduceLROnPlateau`):** Monitors validation accuracy and dynamically decays the learning rate by 50% if the model stalls for 5 epochs. This prevents the optimizer from thrashing and allows it to settle into finer local minima.
* **Automated Generalization Guardrails (`EarlyStopping` & `ModelCheckpoint`):** Halts training if validation accuracy fails to improve after 15 epochs and automatically restores/saves the weights from the absolute best epoch to disk.
* **Gradient Stabilization (The "Double-Steering" Fix):** Explicitly removes standard Keras `class_weight` arguments from the `model.fit()` loop. Class imbalance is now safely and natively handled entirely by the `DynamicFocalLoss`'s internal $\alpha$ array, preventing gradient explosions.
* **Diagnostic Logging (`CSVLogger`):** Records epoch-by-epoch metrics for post-training generalization gap analysis and loss landscape diagnostics.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger, Callback

# Define the callbacks
early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=15, 
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    filepath=os.path.join(MODELS_DIR, 'best_cnn_model_v11.0.keras'), 
    monitor='val_accuracy', 
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',  # Tracks smooth optimization updates
    factor=0.5,        
    patience=5,          
    mode='max', 
    min_lr=1e-6, 
    verbose=1
)

csv_logger = CSVLogger(os.path.join(MODELS_DIR, 'training_log_v11.0.csv'), append=False)


# --- 5. EXECUTION ---
callbacks = [
    early_stopping, 
    model_checkpoint,
    reduce_lr,
    csv_logger,
    DynamicGammaCallback()
]

print(f"Applying targeted class weights: {alpha_weights}")

# Start training
print("--- STARTING MODEL TRAINING ---")
# IMPORTANT: removed class_weight=class_weights to prevent gradient explosion
history = model.fit(
    X_train_final, y_train,
    validation_data=(X_test_final, y_test),
    epochs=100,
    batch_size=32, 
    callbacks=callbacks
)
print("--- TRAINING COMPLETE ---")






# Phase 6: Model Evaluation and Logs Plot




In [ ]:
from src.model_evaluation import plot_training_history, evaluate_model


In [ ]:
evaluate_model(os.path.join(MODELS_DIR, 'best_cnn_model_v9.0.keras'), X_test_final, y_test, os.path.join(MODELS_DIR, 'label_encoder.joblib'))

In [ ]:
plot_training_history (os.path.join(MODELS_DIR, 'training_log_v9.0.csv'))

# Phase 7: Post-Training Quantization (TinyML)

Shrinking the 32-bit Float model to an 8-bit Integer model. Overriding standard TFLite ops to support dynamic LSTM memory (TensorListReserve).

In [ ]:
import tensorflow as tf
import numpy as np
import os

MODELS_DIR = os.path.join("..", "models")
model_path = os.path.join(MODELS_DIR, 'best_cnn_model_v9.0.keras') 
tflite_model_path = os.path.join(MODELS_DIR, 'emotion_model_int8.tflite')

print(f"Loading {os.path.basename(model_path)} Model...")
model = tf.keras.models.load_model(model_path, compile=False)

# 1. Initialize the converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 2. Set optimization to default (required for quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 3. Create a Representative Dataset Generator (Uses your training data to calibrate the 8-bit math)
# Note: Ensure X_train_final is loaded in your environment before running this!
def representative_dataset_gen():
    # Take 100 random samples to calibrate
    for i in range(100):
        # Shape must match input: (1, 60, 125, 1) and be float32
        sample = np.expand_dims(X_train_final[i], axis=0).astype(np.float32)
        yield [sample]

converter.representative_dataset = representative_dataset_gen

# 4. Force strict INT8 conversion and Handle LSTM operations
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS # Required for complex LSTM backend ops
]
converter._experimental_lower_tensor_list_ops = False # Fixes the TensorListReserve error for LSTMs

converter.inference_input_type = tf.int8  # Quantize input
converter.inference_output_type = tf.int8 # Quantize output

print("Quantizing model to INT8 (This may take 1-3 minutes)...")
tflite_model = converter.convert()

# 5. Save the TFLite model
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"SUCCESS! TinyML model saved to: {tflite_model_path}")
print(f"Original Model Size: {os.path.getsize(model_path) / 1024:.2f} KB")
print(f"Quantized Model Size: {os.path.getsize(tflite_model_path) / 1024:.2f} KB")